In [ ]:
from pathlib import Path


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in (current, *current.parents):
        if (path / "data").exists():
            return path
    return current


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"

import os
import random
import re
import threading
import time
from collections import deque
from datetime import datetime, timedelta

import pandas as pd
import tushare as ts

TOKEN = os.getenv("TUSHARE_TOKEN", "").strip()
HTTP_URL = os.getenv("TUSHARE_HTTP_URL", "").strip()
REQUEST_TIMEOUT_SEC = 20
STOCK_LIST_DIR = DATA_ROOT / "data_download" / "tushare_min_download" / "stock_list"
MISSING_DATES_CSV = DATA_ROOT / "data_download" / "tushare_min_download" / "minute_missing_dates.csv"
MISSING_DATE_COL = "date"
MISSING_FILTER_START = "20150101"
MISSING_FILTER_END = "20260101"
OUT_ROOT = DATA_ROOT / "data_download" / "tushare_min_download" / "stock_minute"
WINDOW_DAYS_BEFORE = 3
WINDOW_DAYS_AFTER = 3
ADJ = "hfq"
ADJ_FACTOR = True
FREQ = "1min"
ASSET = "E"
LIMIT = 8000
MAX_OFFSET = 100000
MAX_RETRY = 25
BASE_SLEEP = 0.2
RATE_LIMIT_PER_MIN = 150
RATE_PERIOD_SEC = 60
ENABLE_OVERWRITE = False
PARQUET_ENGINE = "pyarrow"
COMPRESSION = "snappy"

if not TOKEN:
    raise RuntimeError("TUSHARE_TOKEN is required")

pro = ts.pro_api(TOKEN)
if HTTP_URL:
    pro._DataApi__http_url = HTTP_URL
try:
    pro._DataApi__timeout = REQUEST_TIMEOUT_SEC
except Exception:
    pass


class CooldownGate:
    def __init__(self):
        self.lock = threading.Lock()
        self.next_allowed = 0.0

    def set_cooldown(self, seconds):
        with self.lock:
            self.next_allowed = max(self.next_allowed, time.time() + float(seconds))

    def wait(self):
        while True:
            with self.lock:
                next_allowed = self.next_allowed
            now = time.time()
            if now >= next_allowed:
                return
            time.sleep(min(1.0, next_allowed - now))


class RateLimiter:
    def __init__(self, max_calls, period_sec):
        self.max_calls = int(max_calls)
        self.period_sec = float(period_sec)
        self.lock = threading.Lock()
        self.times = deque()

    def acquire(self):
        while True:
            with self.lock:
                now = time.time()
                while self.times and now - self.times[0] >= self.period_sec:
                    self.times.popleft()
                if len(self.times) < self.max_calls:
                    self.times.append(now)
                    return
                wait_sec = self.period_sec - (now - self.times[0])
            time.sleep(max(0.1, wait_sec))


cooldown_gate = CooldownGate()
rate_limiter = RateLimiter(RATE_LIMIT_PER_MIN, RATE_PERIOD_SEC)


def wait_from_message(message):
    match = re.search(r"(\d+)\s*seconds", message.lower())
    if match:
        return int(match.group(1)) + 2
    if "frequency" in message.lower() or "rate" in message.lower():
        return 10
    return None


def read_stock_list():
    frames = []
    if STOCK_LIST_DIR.exists():
        for path in sorted(STOCK_LIST_DIR.glob("*.csv")):
            frames.append(pd.read_csv(path))
        for path in sorted(STOCK_LIST_DIR.glob("*.parquet")):
            frames.append(pd.read_parquet(path))
    if not frames:
        data = pro.stock_basic(exchange="", list_status="L", fields="ts_code,symbol,name,area,industry,list_date")
        return data["ts_code"].dropna().astype(str).sort_values().tolist()
    frame = pd.concat(frames, ignore_index=True)
    col = "ts_code" if "ts_code" in frame.columns else "code"
    values = frame[col].dropna().astype(str).str.strip()
    return sorted(values.unique().tolist())


def read_target_dates():
    if not MISSING_DATES_CSV.exists():
        return []
    frame = pd.read_csv(MISSING_DATES_CSV)
    dates = frame[MISSING_DATE_COL].dropna().astype(str).str.replace("-", "", regex=False).str.slice(0, 8)
    if MISSING_FILTER_START:
        dates = dates[dates >= MISSING_FILTER_START]
    if MISSING_FILTER_END:
        dates = dates[dates <= MISSING_FILTER_END]
    return sorted(dates.unique().tolist())


def date_window(center_day):
    center = datetime.strptime(center_day, "%Y%m%d")
    start = center - timedelta(days=WINDOW_DAYS_BEFORE)
    end = center + timedelta(days=WINDOW_DAYS_AFTER)
    return start.strftime("%Y%m%d"), end.strftime("%Y%m%d")


def fetch_minute(ts_code, start_date, end_date):
    rows = []
    offset = 0
    while offset <= MAX_OFFSET:
        for attempt in range(MAX_RETRY):
            try:
                cooldown_gate.wait()
                rate_limiter.acquire()
                frame = ts.pro_bar(
                    ts_code=ts_code,
                    api=pro,
                    start_date=start_date,
                    end_date=end_date,
                    adj=ADJ,
                    adjfactor=ADJ_FACTOR,
                    freq=FREQ,
                    asset=ASSET,
                    limit=LIMIT,
                    offset=offset,
                )
                if frame is None or frame.empty:
                    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
                rows.append(frame)
                if len(frame) < LIMIT:
                    return pd.concat(rows, ignore_index=True)
                offset += LIMIT
                break
            except Exception as exc:
                wait_sec = wait_from_message(str(exc))
                if wait_sec is not None:
                    cooldown_gate.set_cooldown(wait_sec)
                time.sleep(BASE_SLEEP * (attempt + 1) + random.random() * BASE_SLEEP)
        else:
            raise RuntimeError(f"fetch failed ts_code={ts_code} start={start_date} end={end_date}")
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def write_day(day, codes):
    start_date, end_date = date_window(day)
    out_dir = OUT_ROOT / day
    out_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for code in codes:
        out_path = out_dir / f"{code}.parquet"
        if out_path.exists() and not ENABLE_OVERWRITE:
            continue
        frame = fetch_minute(code, start_date, end_date)
        if frame.empty:
            continue
        frame.to_parquet(out_path, engine=PARQUET_ENGINE, compression=COMPRESSION, index=False)
        count += 1
    print(f"day={day} files={count} output={out_dir}")


def main():
    codes = read_stock_list()
    days = read_target_dates()
    if not days:
        raise RuntimeError(f"missing date file is empty or missing: {MISSING_DATES_CSV}")
    for day in days:
        write_day(day, codes)
    print(f"done output={OUT_ROOT}")


if __name__ == "__main__":
    main()